# Task 3 - Data Preprocessing and Feature Engineering

This notebook improves the quality of the dataset for eLCS while avoiding data leakage.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
# Start from the original dataset so Task 3 is reproducible
data = pd.read_csv("../data/patient_data.csv")

print("Dataset shape:", data.shape)
print("Missing values:", data.isna().sum().sum())
print("Exact duplicate rows:", data.duplicated().sum())

Dataset shape: (280985, 39)
Missing values: 0
Exact duplicate rows: 2284


## 3.1 Target and Leakage Prevention


In [3]:
classLabel = "heart_disease_target"

# HT in sublabel represents a record containing heart disease
data[classLabel] = (
    data["sublabel"]
    .str.contains("HT", na=False)
    .astype(int)
)

print(data[classLabel].value_counts())
print()
print(data[classLabel].value_counts(normalize=True))

heart_disease_target
0    275081
1      5904
Name: count, dtype: int64

heart_disease_target
0    0.978988
1    0.021012
Name: proportion, dtype: float64


## 3.2 Exact Duplicate Records


In [4]:
rows_before = len(data)

# Remove complete duplicate records
data_clean = data.drop_duplicates().copy()

rows_after = len(data_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Remaining exact duplicates:", data_clean.duplicated().sum())

Rows before: 280985
Rows after: 278701
Rows removed: 2284
Remaining exact duplicates: 0


In [5]:
leakage_columns = [
    "sublabel",
    "disease_flags",
    "label",
    "heart_disease"
]

X = data_clean.drop(
    columns=leakage_columns + [classLabel]
)

y = data_clean[classLabel]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nPositive rate:")
print(y.mean())

Predictor shape: (278701, 35)
Target shape: (278701,)

Target distribution:
heart_disease_target
0    272810
1      5891
Name: count, dtype: int64

Positive rate:
0.021137347910484713


In [6]:
# Check whether multiple records now share the same predictor values
duplicate_feature_rows = X.duplicated(
    keep=False
)

print(
    "Rows sharing predictor values:",
    duplicate_feature_rows.sum()
)

print(
    "Unique repeated predictor profiles:",
    X[duplicate_feature_rows]
    .drop_duplicates()
    .shape[0]
)

Rows sharing predictor values: 524
Unique repeated predictor profiles: 262


In [7]:
profile_check = X.copy()
profile_check[classLabel] = y.values

target_counts_per_profile = (
    profile_check
    .groupby(list(X.columns))[classLabel]
    .nunique()
)

conflicting_profiles = target_counts_per_profile[
    target_counts_per_profile > 1
]

print(
    "Predictor profiles with conflicting targets:",
    len(conflicting_profiles)
)

Predictor profiles with conflicting targets: 262


In [8]:
SEED = 3

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

Training shape: (222960, 35)
Testing shape: (55741, 35)

Training positive rate: 0.02113832077502691
Testing positive rate: 0.021133456522129133


## 3.3 Missing Values and Data Types

Missing values and data types are checked again after splitting the cleaned dataset. Any preprocessing rule that must learn information from the data will be fitted using the training set only.

In [9]:
print("Training missing values:", X_train.isna().sum().sum())
print("Testing missing values:", X_test.isna().sum().sum())

print("\nTraining data types:")
print(X_train.dtypes.value_counts())

Training missing values: 0
Testing missing values: 0

Training data types:
float64    18
str        16
int64       1
Name: count, dtype: int64


In [10]:
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:", len(categorical_columns))

for column in categorical_columns:
    print(f"\n{column}:")
    print(sorted(
        X_train[column]
        .dropna()
        .astype(str)
        .unique()
    ))

Categorical columns: 16

composite_key:
['Mature_Female_Normal_Current_No', 'Mature_Female_Normal_Current_Yes', 'Mature_Female_Normal_Former_No', 'Mature_Female_Normal_Former_Yes', 'Mature_Female_Normal_Never_No', 'Mature_Female_Normal_Never_Yes', 'Mature_Female_Obese_Current_No', 'Mature_Female_Obese_Current_Yes', 'Mature_Female_Obese_Former_No', 'Mature_Female_Obese_Former_Yes', 'Mature_Female_Obese_Never_No', 'Mature_Female_Obese_Never_Yes', 'Mature_Female_Overweight_Current_No', 'Mature_Female_Overweight_Current_Yes', 'Mature_Female_Overweight_Former_No', 'Mature_Female_Overweight_Former_Yes', 'Mature_Female_Overweight_Never_No', 'Mature_Female_Overweight_Never_Yes', 'Mature_Female_Underweight_Current_No', 'Mature_Female_Underweight_Current_Yes', 'Mature_Female_Underweight_Former_No', 'Mature_Female_Underweight_Former_Yes', 'Mature_Female_Underweight_Never_No', 'Mature_Female_Underweight_Never_Yes', 'Mature_Male_Normal_Current_No', 'Mature_Male_Normal_Current_Yes', 'Mature_Male_Nor

In [11]:
numerical_columns = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

numerical_summary = X_train[
    numerical_columns
].describe().T

numerical_summary

,count,mean,std,min,25%,50%,75%,max
age,222960.0,49.725368,21.682929,2.000000,32.000000,50.000000,68.000000,89.000000
age_normalized,222960.0,0.548567,0.249229,0.000000,0.344827,0.551724,0.758620,1.000000
bmi,222960.0,27.490879,7.067417,10.010000,22.000000,27.320000,32.700000,95.690000
hypertension,222960.0,0.129267,0.187690,0.000000,0.000000,0.071823,0.217063,1.000000
HbA1c_level,222960.0,5.952595,0.952922,3.500000,5.387500,5.800000,6.865766,9.000000
glucose,222960.0,135.266042,38.471530,70.000000,100.000000,137.000000,160.000000,300.000000
cholesterol,222960.0,223.985144,35.681088,150.000000,201.000000,225.000000,243.000000,300.000000
sleep_hours,222960.0,6.954918,1.432134,4.000000,6.200000,6.975185,7.700000,10.000000
triglycerides,222960.0,178.468709,67.321256,50.000000,128.000000,177.000000,241.750000,400.000000
blood_pressure,222960.0,149.890027,4.165678,120.000000,148.946429,149.713320,150.720942,180.000000
